# Alpha Model Demo

This notebook demonstrates a complete alpha model pipeline from data to signals.
Enhanced with purged/embargoed CV, transaction costs, and ML factor combination.


In [ ]:
# Setup & Configurationimport sysimport subprocessimport warningsimport oswarnings.filterwarnings('ignore')def install(pkg):    try:        __import__(pkg)    except ImportError:        print(f"Installing {pkg}...")        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])# Core dependenciesrequired_packages = ['yfinance', 'lightgbm', 'pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']for pkg in required_packages:    install(pkg)import yfinance as yfimport pandas as pdimport numpy as npimport lightgbm as lgbfrom scipy import statsfrom sklearn.model_selection import TimeSeriesSplitimport matplotlib.pyplot as pltfrom datetime import datetime# Environment detectionIN_KAGGLE = 'KAGGLE_URL_BASE' in os.environIN_COLAB = 'google.colab' in sys.modulesIN_LOCAL = not (IN_KAGGLE or IN_COLAB)env_name = 'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'print(f"🚀 Running in {env_name} environment")print(f"🐍 Python {sys.version.split()[0]}")# Configuration block - easily modifiableTICKERS = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'GLD', 'TLT', 'BTC-USD', 'ETH-USD', 'SOL-USD']START = '2020-01-01'END = '2024-12-31'HORIZON = 5  # Forward return horizon in daysN_SPLITS = 4  # Number of CV foldsSEED = 42     # Random seed for reproducibilityTRANSACTION_COST = 0.001  # 0.1% commission per tradeSLIPPAGE = 0.0005         # 0.05% slippage per tradeOOS_START = '2024-06-01'  # Out-of-sample start datePURGED_EMBARGO = 21       # Days to embargo/purge for CV (1 month)print("\n📋 Configuration:")print(f"   TICKERS: {TICKERS}")print(f"   START: {START}")print(f"   END: {END}")print(f"   HORIZON: {HORIZON}")print(f"   N_SPLITS: {N_SPLITS}")print(f"   SEED: {SEED}")print(f"   TRANSACTION_COST: {TRANSACTION_COST}")print(f"   SLIPPAGE: {SLIPPAGE}")print(f"   OOS_START: {OOS_START}")print(f"   PURGED_EMBARGO: {PURGED_EMBARGO}")

## 1. Data Download

Downloading adjusted close prices for our universe of tickers using Yahoo Finance.

In [ ]:
# Download dataprint(f"📥 Downloading data for {len(TICKERS)} tickers from {START} to {END}...")data_raw = yf.download(    tickers=TICKERS,    start=START,    end=END,    auto_adjust=True,    progress=False)['Close']# Handle single ticker caseif len(TICKERS) == 1:    data_raw = data_raw.to_frame()    data_raw.columns = TICKERS# Forward fill and drop columns with all NaNdata = data_raw.ffill().dropna(axis=1, how='all')# Calculate returnsreturns = data.pct_change()print(f"   Loaded {len(data.columns)} assets, {len(data)} trading days")print(f"   Date range: {data.index[0].date()} to {data.index[-1].date()}")# Show basic statisticsdisplay(data.describe().T)

## 2. Factor Engineering

Computing five core factors that will be combined into our alpha signal:

1. **Momentum (12-1)**: 12-month return skipping the most recent month (Jegadeesh & Titman, 1993)
2. **Reversal (1-month)**: Negative of 1-month return (short-term mean reversion)
3. **Volatility (60-day)**: 60-day rolling standard deviation of returns (annualized)
4. **Skewness (60-day)**: 60-day rolling skewness of returns
5. **Trend (20-day)**: Slope of daily linear regression over 20 days (trend strength)

All factors are cross-sectionally ranked (z-scored) within each date to create market-neutral signals.

In [ ]:
# Compute factorsprint("🔧 Computing factors...")# Initialize factor dataframefactors = pd.DataFrame(index=data.index, columns=data.columns)# 1. Momentum (12-1): 12-month return skipping last monthmom_12_1 = data.pct_change(252).shift(21)  # Approx 12 months - 1 monthfactors['mom_12_1'] = mom_12_1# 2. Reversal (1-month): Negative 1-month returnreversal_1m = -returns.rolling(21).mean()  # Approx 1 monthfactors['reversal_1m'] = reversal_1m# 3. Volatility (60-day): 60-day rolling standard deviationvol_60 = returns.rolling(60).std() * np.sqrt(252)  # Annualizedfactors['vol_60'] = vol_60# 4. Skewness (60-day): 60-day rolling skewnessskew_60 = returns.rolling(60).skew()factors['skew_60'] = skew_60# 5. Trend (20-day): Slope of daily linear regression over 20 daysdef linreg_slope(series):    if len(series) < 2:        return np.nan    x = np.arange(len(series))    slope, _, _, _, _ = stats.linregress(x, series)    return slopetrend_20 = data.rolling(20).apply(linreg_slope, raw=False)factors['trend_20'] = trend_20# Cross-sectional ranking (z-score) for each factorranked_factors = factors.copy()for col in factors.columns:    # Rank within each date (cross-sectional)    ranked = factors[col].rank(axis=1, pct=True)    # Convert to z-scores: (rank - 0.5) * sqrt(12) for approx normal distribution    # For simplicity, we'll use rank - 0.5 giving range [-0.5, 0.5]    ranked_factors[col] = ranked - 0.5# Combined factor signal (equal weight)factor_signal = ranked_factors.mean(axis=1)print(f"   Computed {len(factors.columns)} factors")print(f"   Factor signal range: [{factor_signal.min().min():.3f}, {factor_signal.max().max():.3f}]")# Show factor correlationsfactor_corr = ranked_factors.unstack().groupby(level=0).corr().unstack(level=0)print("\n🔗 Factor correlations:")display(factor_corr.round(2))

## 3. Factor Analysis & Backtest

Analyzing the predictive power of each factor using Information Coefficient (IC)
and running a factor-only backtest with transaction costs.

In [ ]:
# Calculate forward returnsprint("⏩ Calculating forward returns...")# Forward returns aligned with factor calculation date# Factors calculated at time t predict returns from t+1 to t+HORIZONforward_returns = data.pct_change(HORIZON).shift(-HORIZON)print(f"   Forward returns horizon: {HORIZON} days")print(f"   Forward returns date range: {forward_returns.index[0].date()} to {forward_returns.index[-1].date()}")# Align factors and forward returns (factors known at time t)# Shift factors forward by 1 day so they're known at prediction timefactors_for_prediction = factors.shift(1)# Drop NaN valuescombined_data = pd.concat([factors_for_prediction, forward_returns], axis=1).dropna()if len(combined_data) == 0:    # Fallback: use same-day alignment for simplicity    print("   Warning: No overlapping data with shift, using same-day alignment")    factors_for_prediction = factors    combined_data = pd.concat([factors, forward_returns], axis=1).dropna()# Separate factors and returnsfactor_data = combined_data.iloc[:, :len(factors.columns)]return_data = combined_data.iloc[:, len(factors.columns):]# Calculate Information Coefficients (IC) for each factorprint("\n📊 Calculating Information Coefficients (IC)...")ic_values = {}for factor in factors.columns:    # Calculate time-series average of cross-sectional correlations    ic_series = []    for date in factor_data.index:        # Get factor values for all tickers at this date        factor_vals = factor_data.loc[date, factors.columns]        # Get corresponding returns for all tickers at this date        # Note: return_data has same structure as factor_data        return_vals = return_data.loc[date, factors.columns]        # Calculate cross-sectional correlation        if len(factor_vals) > 1 and not factor_vals.isna().all() and not return_vals.isna().all():            ic = np.corrcoef(factor_vals.fillna(0), return_vals.fillna(0))[0, 1]            if not np.isnan(ic):                ic_series.append(ic)        if ic_series:        ic_values[factor] = np.mean(ic_series)    else:        ic_values[factor] = 0.0ic_series = pd.Series(ic_values)print("\n📊 Factor Information Coefficients (IC):")print("   (Time-series average of cross-sectional IC)")for factor, ic in ic_series.items():    print(f"   {factor:10s}: {ic:+.4f} {'✅' if abs(ic) > 0.02 else '⚠️'}")# Factor returns backtestprint("\n📈 Factor-only backtest (equal weight long/short):")# For each date, go long top-ranked factors, short bottom-ranked# We'll use the factor signal directly as position size (market neutral)factor_position = factor_signal.clip(-1, 1)  # Limit to [-1, 1] for diversification# Calculate returns: position known at t-1, return from t-1 to tfactor_returns = (factor_position.shift(1) * returns).sum(axis=1)  # Sum across all tickers# Apply transaction costsposition_changes = factor_position.diff().abs()  # Absolute change in positiontransaction_costs = (position_changes * TRANSACTION_COST).sum(axis=1)  # Cost per tradeslippage_costs = (position_changes * SLIPPAGE).sum(axis=1)  # Slippage per tradefactor_returns_net = factor_returns - transaction_costs - slippage_costs# Calculate cumulative returnsfactor_cumulative = (1 + factor_returns_net.fillna(0)).cumprod()# Calculate performance metricstotal_return = factor_cumulative.iloc[-1] - 1trading_days = len(factor_returns_net)annual_return = (1 + total_return) ** (252 / trading_days) - 1 if trading_days > 0 else 0annual_vol = factor_returns_net.std() * np.sqrt(252) if trading_days > 1 else 0sharpe = annual_return / annual_vol if annual_vol > 0 else 0# Sortino ratio (downside deviation)downside_returns = factor_returns_net[factor_returns_net < 0]downside_deviation = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 1 else 0sortino = annual_return / downside_deviation if downside_deviation > 0 else 0# Calmar ratio (annual return / max drawdown)rolling_max = factor_cumulative.cummax()drawdown = (factor_cumulative / rolling_max) - 1max_drawdown = drawdown.min()calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0# Win ratewins = (factor_returns_net > 0).sum()total_non_zero = (factor_returns_net != 0).sum()win_rate = wins / total_non_zero if total_non_zero > 0 else 0print(f"   Total Return: {total_return:+.2%}")print(f"   Annual Return: {annual_return:+.2%}")print(f"   Annual Volatility: {annual_vol:.2%}")print(f"   Sharpe Ratio: {sharpe:.2f}")print(f"   Sortino Ratio: {sortino:.2f}")print(f"   Calmar Ratio: {calmar:.2f}")print(f"   Max Drawdown: {max_drawdown:.2%}")print(f"   Win Rate: {win_rate:.2%}")print(f"   Total Trading Days: {trading_days}")# Plot factor performancefig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))# Equity curveax1.plot(factor_cumulative.index, factor_cumulative.values, linewidth=2, color='darkblue')ax1.set_title('Factor Model Equity Curve', fontsize=14, fontweight='bold')ax1.set_ylabel('Cumulative Return')ax1.grid(True, alpha=0.3)ax1.legend(['Factor Model'])# Drawdownax2.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)ax2.set_title('Drawdown', fontsize=14, fontweight='bold')ax2.set_ylabel('Drawdown')ax2.set_xlabel('Date')ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 4. ML Feature Engineering

Creating machine learning features from price and volume data.

Features include:
- Returns: 5, 10, 20, 60, 120 day returns
- Volatility: 5, 10, 20, 60, 120 day volatility
- Risk-adjusted returns: Sharpe-like ratios for each lookback
- All features are cross-sectionally ranked (z-scored) within each date


In [ ]:
# Create ML featuresprint("🤖 Creating ML features...")def create_ml_features(prices):    """Create ML features from price data."""    feat = pd.DataFrame(index=prices.index)    r = prices.pct_change()        # Returns and volatilities for multiple lookbacks    lookbacks = [5, 10, 20, 60, 120]    for lb in lookbacks:        feat[f'ret_{lb}d'] = r.rolling(lb).sum()        feat[f'vol_{lb}d'] = r.rolling(lb).std() * np.sqrt(252)  # Annualized        feat[f'sharpe_{lb}d'] = feat[f'ret_{lb}d'] / (feat[f'vol_{lb}d'] + 1e-8)  # Avoid division by zero        # Rank features cross-sectionally (z-score)    for col in feat.columns:        ranked = feat[col].rank(axis=1, pct=True)        feat[col] = ranked - 0.5  # Range [-0.5, 0.5]        return feat.dropna()ml_features = create_ml_features(data)print(f"   Created {len(ml_features.columns)} ML features")print(f"   Feature data shape: {ml_features.shape}")# Show feature correlations (first 5 features)if len(ml_features.columns) >= 5:    feature_corr = ml_features.iloc[:, :5].corr()    print("\n🔗 Sample feature correlations (first 5):")    display(feature_corr.round(2))# Prepare data for ML: features predict forward returnsprint("\n⏩ Preparing ML dataset...")# Features at time t predict forward returns from t+1 to t+HORIZONml_features_lagged = ml_features.shift(1)  # Features known at previous dayforward_returns_ml = data.pct_change(HORIZON).shift(-HORIZON)  # Future returns# Stack data for sklearnfeatures_stacked = ml_features_lagged.stack()returns_stacked = forward_returns_ml.stack()# Align indicescommon_index = features_stacked.index.intersection(returns_stacked.index)X = features_stacked.loc[common_index]y = returns_stacked.loc[common_index]# Remove NaN valuesvalid_idx = ~(X.isna().any(axis=1) | y.isna())X = X[valid_idx]y = y[valid_idx]print(f"   ML dataset: {X.shape[0]} samples, {X.shape[1]} features")print(f"   Date range: {X.index.get_level_values(0)[0].date()} to {X.index.get_level_values(0)[-1].date()}")

## 5. Time-Series Cross-Validation

Using Purged/Embargoed Cross-Validation to prevent lookahead bias in financial time series.

**Why Purged/Embargoed CV?**
- Standard CV can leak future information into training
- Purging removes samples close to the test period in time
- Embargoing adds a buffer period after the test period
- This ensures no temporal leakage between train and test sets
**Configuration**: 4-fold CV with 21-day purge/embargo

In [ ]:
# Purged/Embargoed Time-Series Cross-Validationprint("🔄 Running Purged/Embargoed Time-Series Cross-Validation...")def purged_embargoed_cv(X, y, n_splits=5, purge_days=0, embargo_days=0):    """Generate train/test indices for purged/embargoed CV."""    n_samples = len(X)    indices = np.arange(n_samples)    test_size = n_samples // (n_splits + 1)  # Approximate equal sizes        test_starts = [i * test_size for i in range(n_splits)]    test_ends = [(i + 1) * test_size for i in range(n_splits)]        for i in range(n_splits):        test_start = test_starts[i]        test_end = test_ends[i]                # Test indices        test_indices = indices[test_start:test_end]                # Train indices: everything before test period (with embargo) and after test period (with purge)        train_before_end = max(0, test_start - embargo_days)        train_after_start = test_end + purge_days                train_indices = np.concatenate([            indices[0:train_before_end],            indices[train_after_start:n_samples]        ])                yield train_indices, test_indices# Run CVics = []models = []fold_details = []cv_splitter = purged_embargoed_cv(X, y, n_splits=N_SPLITS, purge_days=PURGED_EMBARGO, embargo_days=PURGED_EMBARGO)for fold, (train_idx, test_idx) in enumerate(cv_splitter, 1):    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]        # Train LightGBM model    model = lgb.LGBMRegressor(        n_estimators=200,        max_depth=5,        learning_rate=0.05,        subsample=0.8,        colsample_bytree=0.8,        reg_alpha=0.1,        reg_lambda=0.1,        random_state=SEED + fold,        n_jobs=-1,        verbosity=-1    )        model.fit(X_train, y_train)        # Predict and calculate IC    y_pred = model.predict(X_test)    ic = np.corrcoef(y_pred, y_test)[0, 1] if len(y_pred) > 1 else 0    ics.append(ic)    models.append(model)        fold_details.append({        'fold': fold,        'train_size': len(X_train),        'test_size': len(X_test),        'ic': ic,        'train_date_range': f"{X_train.index.get_level_values(0)[0].date()} to {X_train.index.get_level_values(0)[-1].date()}",        'test_date_range': f"{X_test.index.get_level_values(0)[0].date()} to {X_test.index.get_level_values(0)[-1].date()}"    })        print(f"   Fold {fold}: IC = {ic:+.4f} (Train: {len(X_train)}, Test: {len(X_test)})")print(f"\n📊 Cross-Validation Results:")print(f"   Mean IC: {np.mean(ics):+.4f} ± {np.std(ics):.4f}")print(f"   IC Range: [{np.min(ics):+.4f}, {np.max(ics):+.4f}]")# Show fold detailsfold_df = pd.DataFrame(fold_details)display(fold_df[['fold', 'train_size', 'test_size', 'ic', 'train_date_range', 'test_date_range']])

## 6. Final Model Training & Today's Signal

Training the final model on all available data and generating today's trading signal.

In [ ]:
# Train final model on all dataprint("🎯 Training final model on all data...")final_model = lgb.LGBMRegressor(    n_estimators=200,    max_depth=5,    learning_rate=0.05,    subsample=0.8,    colsample_bytree=0.8,    reg_alpha=0.1,    reg_lambda=0.1,    random_state=SEED,    n_jobs=-1,    verbosity=-1)final_model.fit(X, y)# Feature importancefeature_importance = pd.DataFrame({    'feature': X.columns,    'importance': final_model.feature_importances_}).sort_values('importance', ascending=False)print(f"\n🔝 Top 10 Most Important Features:")display(feature_importance.head(10))# Generate today's signal (most recent data point)print("\n📡 Generating today's trading signal...")# Get most recent features for each ticker# We need to get the latest features for each tickerlatest_features = ml_features.groupby(ml_features.index.get_level(1)).last()# Align with feature nameslatest_features_aligned = latest_features.reindex(columns=X.columns, fill_value=0)# Generate predictionstoday_signal_raw = final_model.predict(latest_features_aligned)today_signal = pd.Series(today_signal_raw, index=latest_features.index)# Convert to ranked signal (long/short)today_signal_ranked = today_signal.rank(pct=True) - 0.5  # Range [-0.5, 0.5]print(f"   Generated signals for {len(today_signal)} tickers")print(f"   Signal range: [{today_signal_ranked.min():+.3f}, {today_signal_ranked.max():+.3f}]")# Show today's signalstoday_signals_df = pd.DataFrame({    'raw_signal': today_signal,    'ranked_signal': today_signal_ranked,    'position': today_signal_ranked.clip(-1, 1)  # Limit position size}).sort_values('ranked_signal', ascending=False)display(today_signals_df.head(10))# Count longs and shortsn_long = (today_signals_df['ranked_signal'] > 0).sum()n_short = (today_signals_df['ranked_signal'] < 0).sum()n_neutral = (today_signals_df['ranked_signal'] == 0).sum()print(f"\n📊 Signal Summary:")print(f"   Longs:  {n_long:3d} 🟢")print(f"   Shorts: {n_short:3d} 🔴")print(f"   Neutral:{n_neutral:3d} ⚪")

## 7. Combined Signal (Factor + ML Ensemble)

Combining the factor signal and ML signal into a single alpha signal.

Approaches for combining signals:
- Equal weight average
- Weighted by historical IC (more predictive signal gets higher weight)
- Regime-dependent weighting (different weights in different market regimes)

We'll use equal weight for simplicity, but show how to implement IC-weighted combination.

In [ ]:
# Generate combined signalprint("🔗 Generating combined signal (Factor + ML)...")# Get latest factor signal for each ticker# Factors are calculated at time t, so we use yesterday's factors to predict todaylatest_factors = factors.groupby(factors.index.get_level(1)).last()latest_factors_aligned = latest_factors.reindex(columns=factors.columns, fill_value=0)# Calculate factor signal (same as before)ranked_factors_today = latest_factors_aligned.copy()for col in latest_factors_today.columns:    ranked = latest_factors_today[col].rank(pct=True)    ranked_factors_today[col] = ranked - 0.5factor_signal_today = ranked_factors_today.mean(axis=1)factor_signal_today = pd.Series(factor_signal_today, index=latest_factors.index)# Ensure we have the same tickers in both signalscommon_tickers = factor_signal_today.index.intersection(today_signal.index)factor_signal_aligned = factor_signal_today[common_tickers]ml_signal_aligned = today_signal[common_tickers]# Method 1: Equal weight combinationcombined_signal_equal = (factor_signal_aligned + ml_signal_aligned) / 2# Method 2: IC-weighted combination (using historical ICs)# We'll use the absolute IC as weight (higher absolute IC = higher weight)factor_weights = np.abs(ic_series.reindex(factors.columns).fillna(0))# For simplicity, we'll use average factor weight (since we don't have per-factor IC today)avg_factor_weight = np.mean(factor_weights) if len(factor_weights) > 0 else 1.0ml_weight = 1.0  # Assume ML weight of 1.0 for simplicity# Normalize weights to sum to 1total_weight = avg_factor_weight + ml_weightfactor_weight_norm = avg_factor_weight / total_weightml_weight_norm = ml_weight / total_weightcombined_signal_weighted = (factor_weight_norm * factor_signal_aligned) + (ml_weight_norm * ml_signal_aligned)print(f"   Factor weight: {factor_weight_norm:.3f}")print(f"   ML weight:     {ml_weight_norm:.3f}")# Use equal weight for final signal (can be changed)combined_signal = combined_signal_equal# Convert to positionspositions = combined_signal.clip(-1, 1)  # Limit to [-1, 1]print(f"\n📊 Combined Signal Summary:")print(f"   Signals generated: {len(combined_signal)} tickers")print(f"   Signal range: [{combined_signal.min():+.3f}, {combined_signal.max():+.3f}]")print(f"   Position range: [{positions.min():+.3f}, {positions.max():+.3f}]")# Show top signalscombined_df = pd.DataFrame({    'factor_signal': factor_signal_aligned,    'ml_signal': ml_signal_aligned,    'combined_signal': combined_signal,    'position': positions}).sort_values('combined_signal', ascending=False)display(combined_df.head(10))# Signal summaryn_long = (positions > 0).sum()n_short = (positions < 0).sum()n_neutral = (positions == 0).sum()print(f"\n📊 Position Summary:")print(f"   Longs:  {n_long:3d} 🟢")print(f"   Shorts: {n_short:3d} 🔴")print(f"   Neutral:{n_neutral:3d} ⚪")

## 8. Performance Analysis & Visualization

Analyzing the performance of our alpha signal and visualizing results.

In [ ]:
# Backtest the combined signalprint("📈 Backtesting combined signal...")# We need to generate historical combined signals for backtesting# For simplicity in this demo, we'll approximate using the same approach# In practice, you'd want to store historical signals or recompute themprint("   Note: For a full backtest, you would need to recompute signals historically")print("   This demo shows today's signal only. For backtest, see the factor-only results above.")# Create a simple performance summaryprint("\n📊 Today's Alpha Signal Summary:")# Calculate signal statisticssignal_mean = combined_signal.mean()signal_std = combined_signal.std()signal_skew = combined_signal.skew()signal_kurt = combined_signal.kurtosis()print(f"   Mean signal: {signal_mean:+.4f}")print(f"   Signal Std:  {signal_std:.4f}")print(f"   Signal Skew: {signal_skew:+.4f}")print(f"   Signal Kurt: {signal_kurt:+.4f}")# Signal distributionfig, axes = plt.subplots(2, 2, figsize=(12, 10))# Signal histogramaxes[0, 0].hist(combined_signal, bins=30, alpha=0.7, color='steelblue', edgecolor='black')axes[0, 0].set_title('Signal Distribution')axes[0, 0].set_xlabel('Signal Value')axes[0, 0].set_ylabel('Frequency')axes[0, 0].grid(True, alpha=0.3)# Signal vs Factor signalaxes[0, 1].scatter(factor_signal_aligned, combined_signal, alpha=0.6, s=20)axes[0, 1].set_xlabel('Factor Signal')axes[0, 1].set_ylabel('Combined Signal')axes[0, 1].set_title('Combined vs Factor Signal')axes[0, 1].grid(True, alpha=0.3)# Signal vs ML signalaxes[1, 0].scatter(ml_signal_aligned, combined_signal, alpha=0.6, s=20, color='orange')axes[1, 0].set_xlabel('ML Signal')axes[1, 0].set_ylabel('Combined Signal')axes[1, 0].set_title('Combined vs ML Signal')axes[1, 0].grid(True, alpha=0.3)# Position distributionposition_counts = [n_long, n_short, n_neutral]position_labels = ['Longs (🟢)', 'Shorts (🔴)', 'Neutral (⚪)']colors = ['green', 'red', 'gray']axes[1, 1].pie(position_counts, labels=position_labels, colors=colors, autopct='%1.1f%%', startangle=90)axes[1, 1].set_title('Position Distribution')plt.tight_layout()plt.show()# Save signals to CSVoutput_df = pd.DataFrame({    'ticker': combined_signal.index,    'factor_signal': factor_signal_aligned.values,    'ml_signal': ml_signal_aligned.values,    'combined_signal': combined_signal.values,    'position': positions.values})output_file = 'alpha_signals_today.csv'output_df.to_csv(output_file, index=False)print(f"\n💾 Signals saved to: {output_file}")print(f"   Shape: {output_df.shape}")# Show top 10 signalsprint(f"\n🏆 Top 10 Signals:")display(output_df.head(10))

## 9. Conclusion & Next Steps

## 🎯 Key Takeaways

1. **Factor Signals**: Traditional factors still provide valuable predictive information
2. **ML Enhancement**: Machine learning can capture non-linear patterns missed by linear factors
3. **Ensemble Benefits**: Combining factor and ML signals often outperforms either alone
4. **Robustness**: Proper validation (purged/embargoed CV) prevents overfitting
5. **Risk Management**: Transaction costs and position limits are crucial for realistic results

## 🔧 Potential Improvements

### Data & Features
- Add alternative data (funding rates, open interest, sentiment)
- Include volume-based factors
- Add sector/industry neutralization

### Model Improvements
- Try different ML algorithms (XGBoost, CatBoost, Neural Networks)
- Add feature selection and regularization
- Implement regime-dependent model weights

### Execution Improvements
- Add execution algorithms (TWAP, VWAP, Implementation Shortfall)
- Include market impact models
- Add portfolio construction optimizations

## 📦 Files Generated

- `alpha_signals_today.csv`: Today's trading signals

## 🚀 Next Steps

1. **Run as scheduled job**: Generate signals daily/weekly
2. **Paper trading**: Test signals in paper trading before live deployment
3. **Live deployment**: Connect to broker API for automated execution
4. **Monitoring**: Track performance, risk metrics, and signal decay
5. **Research**: Continuously improve factors and models

---
*Alpha Model Demo - Generated on """
    + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + """*
